In [1]:
import sys 
sys.path.append('src')
from sr_model import *

import os
from datasets import load_from_disk, concatenate_datasets
from datasets import Dataset
import numpy as np 
import pandas as pd 
from torch.utils.data import DataLoader 

In [2]:
data_path = '../sr_project/eval_data/merge_dataset/mdata.h5mu'
import muon as mu 
mdata = mu.read_h5mu(data_path)

train_idx = np.load('../sr_project/eval_data/data_split/train_id_2.npy')
test_idx = np.load('../sr_project/eval_data/data_split/test_id_2.npy')
print(len(train_idx), len(test_idx))

/home/rsun@ZHANGroup.local/anaconda3/envs/snapatac/lib/python3.10/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/rsun@ZHANGroup.local/anaconda3/envs/snapatac/lib/python3.10/site-packages/mudata/_core/mudata.py:931: UserWarning: Cannot join columns with the same name because var_names are intersecting.
  warnings.warn(


60895 6767


/home/rsun@ZHANGroup.local/anaconda3/envs/snapatac/lib/python3.10/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


In [3]:
train_obs = mdata['rna_count'].obs.index[train_idx]
test_obs = mdata['rna_count'].obs.index[test_idx]
print(len(train_obs),len(test_obs), mdata.shape)

60895 6767 (78886, 696445)


## 10x multi 数据预处理

In [4]:
idx = mdata.obs.loc[:,'rna_count:tissues'] != '10x_multi'
rna = mdata['rna_count']
rna = rna[idx,:]
atac = mdata['peak_count']
atac = atac[idx,:]

mdata = MuData({'rna_count': rna, 'peak_count': atac})

/home/rsun@ZHANGroup.local/anaconda3/envs/snapatac/lib/python3.10/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)


/home/rsun@ZHANGroup.local/anaconda3/envs/snapatac/lib/python3.10/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


In [5]:
import scanpy as sc 
print(rna.shape)
sc.pp.filter_genes(rna, min_cells = int(rna.shape[0] * 0.01))
print(rna.shape) 

sc.pp.normalize_total(rna, target_sum = 1e4)
sc.pp.log1p(rna) 


print(atac.shape)
sc.pp.filter_genes(atac, min_cells = int(atac.shape[0] * 0.01))
print(atac.shape)

sc.pp.normalize_total(atac, target_sum =1e4 )
sc.pp.log1p(atac)

(67662, 32285)


/home/rsun@ZHANGroup.local/anaconda3/envs/snapatac/lib/python3.10/site-packages/scanpy/preprocessing/_simple.py:284: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var["n_cells"] = number


(67662, 14108)
(67662, 608869)


/home/rsun@ZHANGroup.local/anaconda3/envs/snapatac/lib/python3.10/site-packages/scanpy/preprocessing/_simple.py:284: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var["n_cells"] = number


(67662, 143989)


In [6]:
# train_rna = rna[train_idx,:]
# test_rna = rna[test_idx,:]

# train_atac = atac[train_idx,:]
# test_atac = atac[test_idx,:]

# train_mdata = MuData({'rna': train_rna, 'atac': train_atac})
# test_mdata = MuData({'rna': test_rna, 'atac': test_atac})

## 设置pair_sr模型

In [7]:
train_pos = mdata['rna_count'].obs.index.get_indexer(train_obs)
test_pos = mdata['rna_count'].obs.index.get_indexer(test_obs)

print(len(train_obs),len(test_obs))

60895 6767


In [8]:
feature_num_1 = rna.shape[1]
feature_num_2 = atac.shape[1]


hidden_params_1 = [1024,256]
hidden_params_2 = [1024,256]
embed_dim = 256
use_rmsnorm = True
use_residual = True 
dropout_p = 0
clip_temperature = 0.07


pair_model = pair_sr_scratch(feature_num_1, feature_num_2, hidden_params_1, hidden_params_2, embed_dim, use_rmsnorm, use_residual, dropout_p, clip_temperature)



In [9]:
# pair_model.set_loss(vae_beta_1= 1.0, vae_beta_2 = 1.0, clip_weight =  20, cross_recon_1 = 0.4,cross_recon_2 = 0.4,temperature = 0.07)
# pair_model.loss.clip_loss.logit_scale

In [ ]:
pair_model.create_dataset(mdata, 'rna_count', 'peak_count', train_idx = train_pos, test_idx = test_pos)
#pair_model.set_dataloader(batch_size = 512)

In [ ]:
pair_model.set_dataloader(batch_size = 1024)
pair_model.set_loss(vae_beta_1= 1.0, vae_beta_2 = 1.0, clip_weight =  20, cross_recon_1 = 0.4,cross_recon_2 = 0.4,temperature = 0.07)
pair_model.loss.to('cuda')
pair_model.set_optimizer(lr = 1e-3, warmup_steps=400, steady_1_steps = 1600, cosine_anneal_steps=6000, min_lr = 1e-4)
pair_model.set_project('runs/clip_case_2')
pair_model.train_model( train_steps = 10000 ,eval_points = 80,save_points = 1000,device = 'cuda')

 65%|██████▌   | 39/60 [03:04<02:08,  6.13s/it]

In [32]:
for key in range(1,11):
    print('==========='*10)
    key = int(1000*key)    
    pair_model.init_model(f'/home/rsun@ZHANGroup.local/solid-recover/runs/clip_case_2/models/ckpt_{key}.pth')

    model = pair_model.model 

    model.eval()

    x1 = pair_model.test_dataset.omic_1.to('cuda')
    x2 = pair_model.test_dataset.omic_2.to('cuda')

    z, z_mu, z_logvar, z_embed = model.model_1.get_embedding(x1)
    y,y_mu, y_logvar, y_embed  = model.model_2.get_embedding(x2)

    import anndata as ad 

    N = z_embed.shape[0]

    z_mu = z_mu.detach().cpu().numpy()
    z_embed = z_embed.detach().cpu().numpy()

    y_mu = y_mu.detach().cpu().numpy()
    y_embed = y_embed.detach().cpu().numpy()

    mu = np.concatenate([z_mu, y_mu], axis = 0)
    print(mu.shape)
    embed = np.concatenate([z_embed, y_embed], axis = 0)
    print(embed.shape)

    import numpy as np 
    import pandas as pd 
    import json
    import scanpy as sc

    import matplotlib.pyplot as plt 
    import seaborn as sns 
    import os 
    import sys
    import anndata as ad

    sys.path.append('/home/rsun@ZHANGroup.local/sr_project/src')
    from metrics import calculate_hit_rate, matching_metrics


    ## perform evaluation
    res = {}
    ### calculate top_k hit rate
    tmp = []
    for i in [1,5,10,15,20,30,50,100]:
        res[f'top_{i}_hit'] = calculate_hit_rate(z_mu, y_mu, i, metric = 'euclidean')
    print(res)

    ### calculate metric score
    acc, ms, fs = matching_metrics(x=z_embed, y = y_embed, metric='euclidean')
    res['acc'] = acc 
    res['mathscore'] = ms 
    res['foscttm'] = fs

    print(res)


    ## perform evaluation
    res = {}
    ### calculate top_k hit rate
    tmp = []
    for i in [1,5,10,15,20,30,50,100]:
        res[f'top_{i}_hit'] = calculate_hit_rate(z_mu, y_mu, i, metric = 'cosine')
    print(res)

    ### calculate metric score
    acc, ms, fs = matching_metrics(x=z_embed, y = y_embed, metric='cosine')
    res['acc'] = acc 
    res['mathscore'] = ms 
    res['foscttm'] = fs

    print(res)



/home/rsun@ZHANGroup.local/solid-recover/src/sr_model.py:133: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


(13534, 128)
(13534, 128)
{'top_1_hit': 0.023939707403576177, 'top_5_hit': 0.09501994975616965, 'top_10_hit': 0.1560514260381262, 'top_15_hit': 0.20289640904388948, 'top_20_hit': 0.24375646519875868, 'top_30_hit': 0.30486182946652873, 'top_50_hit': 0.395448500073888, 'top_100_hit': 0.5277818826658786}
{'top_1_hit': 0.023939707403576177, 'top_5_hit': 0.09501994975616965, 'top_10_hit': 0.1560514260381262, 'top_15_hit': 0.20289640904388948, 'top_20_hit': 0.24375646519875868, 'top_30_hit': 0.30486182946652873, 'top_50_hit': 0.395448500073888, 'top_100_hit': 0.5277818826658786, 'acc': 0.01152652595192194, 'mathscore': 0.0119698541238904, 'foscttm': 0.07670345902442932}
{'top_1_hit': 0.02430914733264371, 'top_5_hit': 0.0967932614156938, 'top_10_hit': 0.15915472144229348, 'top_15_hit': 0.2065908083345648, 'top_20_hit': 0.24685976060292597, 'top_30_hit': 0.3090734446578986, 'top_50_hit': 0.4001034431801389, 'top_100_hit': 0.5312546179991133}
{'top_1_hit': 0.02430914733264371, 'top_5_hit': 0.09

/home/rsun@ZHANGroup.local/solid-recover/src/sr_model.py:133: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


(13534, 128)
(13534, 128)
{'top_1_hit': 0.06287867592729422, 'top_5_hit': 0.18294665287424264, 'top_10_hit': 0.26673562878675927, 'top_15_hit': 0.3216344022461948, 'top_20_hit': 0.3613122506280479, 'top_30_hit': 0.4272942219595094, 'top_50_hit': 0.50568937490764, 'top_100_hit': 0.6177774493867297}
{'top_1_hit': 0.06287867592729422, 'top_5_hit': 0.18294665287424264, 'top_10_hit': 0.26673562878675927, 'top_15_hit': 0.3216344022461948, 'top_20_hit': 0.3613122506280479, 'top_30_hit': 0.4272942219595094, 'top_50_hit': 0.50568937490764, 'top_100_hit': 0.6177774493867297, 'acc': 0.01381705328822136, 'mathscore': 0.014334268867969513, 'foscttm': 0.09923522174358368}
{'top_1_hit': 0.06443032362937787, 'top_5_hit': 0.18634550022166396, 'top_10_hit': 0.2700605881483671, 'top_15_hit': 0.32488547362198905, 'top_20_hit': 0.36663218560662036, 'top_30_hit': 0.4330574848529629, 'top_50_hit': 0.5120437416876016, 'top_100_hit': 0.6260529037978425}
{'top_1_hit': 0.06443032362937787, 'top_5_hit': 0.1863455

/home/rsun@ZHANGroup.local/solid-recover/src/sr_model.py:133: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


...

(13534, 128)
(13534, 128)
{'top_1_hit': 0.0812767843948574, 'top_5_hit': 0.2043741687601596, 'top_10_hit': 0.28535540121176295, 'top_15_hit': 0.33781587113935274, 'top_20_hit': 0.373873208216344, 'top_30_hit': 0.43335303679621695, 'top_50_hit': 0.511526525786907, 'top_100_hit': 0.6123836264223437}
{'top_1_hit': 0.0812767843948574, 'top_5_hit': 0.2043741687601596, 'top_10_hit': 0.28535540121176295, 'top_15_hit': 0.33781587113935274, 'top_20_hit': 0.373873208216344, 'top_30_hit': 0.43335303679621695, 'top_50_hit': 0.511526525786907, 'top_100_hit': 0.6123836264223437, 'acc': 0.01581203006207943, 'mathscore': 0.016550909727811813, 'foscttm': 0.10446930676698685}
{'top_1_hit': 0.08268065612531403, 'top_5_hit': 0.20976799172454558, 'top_10_hit': 0.2901581202896409, 'top_15_hit': 0.3459435495788385, 'top_20_hit': 0.38318309442884585, 'top_30_hit': 0.44465789862568345, 'top_50_hit': 0.5208364119994089, 'top_100_hit': 0.6243534801241318}
{'top_1_hit': 0.08268065612531403, 'top_5_hit': 0.2097679

/home/rsun@ZHANGroup.local/solid-recover/src/sr_model.py:133: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


(13534, 128)
(13534, 128)
{'top_1_hit': 0.0791340328062657, 'top_5_hit': 0.2009014334269248, 'top_10_hit': 0.27619329097088813, 'top_15_hit': 0.32924486478498594, 'top_20_hit': 0.36788828136545, 'top_30_hit': 0.42581646224323927, 'top_50_hit': 0.5025121915176592, 'top_100_hit': 0.6081720112309739}
{'top_1_hit': 0.0791340328062657, 'top_5_hit': 0.2009014334269248, 'top_10_hit': 0.27619329097088813, 'top_15_hit': 0.32924486478498594, 'top_20_hit': 0.36788828136545, 'top_30_hit': 0.42581646224323927, 'top_50_hit': 0.5025121915176592, 'top_100_hit': 0.6081720112309739, 'acc': 0.01403871737420559, 'mathscore': 0.015220925211906433, 'foscttm': 0.10809488222002983}
{'top_1_hit': 0.08053790453672233, 'top_5_hit': 0.20563026451898922, 'top_10_hit': 0.2835082015664253, 'top_15_hit': 0.33929363085562286, 'top_20_hit': 0.378971479237476, 'top_30_hit': 0.4362346682429437, 'top_50_hit': 0.5162553568789715, 'top_100_hit': 0.6242057041525048}
{'top_1_hit': 0.08053790453672233, 'top_5_hit': 0.205630264

/home/rsun@ZHANGroup.local/solid-recover/src/sr_model.py:133: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


(13534, 128)
(13534, 128)
{'top_1_hit': 0.07839515294813064, 'top_5_hit': 0.19528594650509828, 'top_10_hit': 0.2734594354957884, 'top_15_hit': 0.32695433722476724, 'top_20_hit': 0.36197724250036944, 'top_30_hit': 0.42027486330722624, 'top_50_hit': 0.4966011526525787, 'top_100_hit': 0.6032215161814689}
{'top_1_hit': 0.07839515294813064, 'top_5_hit': 0.19528594650509828, 'top_10_hit': 0.2734594354957884, 'top_15_hit': 0.32695433722476724, 'top_20_hit': 0.36197724250036944, 'top_30_hit': 0.42027486330722624, 'top_50_hit': 0.4966011526525787, 'top_100_hit': 0.6032215161814689, 'acc': 0.014408157207071781, 'mathscore': 0.014186493121087551, 'foscttm': 0.11225869879126549}
{'top_1_hit': 0.07869070489138466, 'top_5_hit': 0.2009014334269248, 'top_10_hit': 0.2803310181764445, 'top_15_hit': 0.3348603517068125, 'top_20_hit': 0.3723215605142604, 'top_30_hit': 0.4321708290232008, 'top_50_hit': 0.5098271021131964, 'top_100_hit': 0.6178513373725433}
{'top_1_hit': 0.07869070489138466, 'top_5_hit': 0.2

In [ ]:
adata = ad.AnnData(X = np.random.rand(2*N,10))
adata.obs.loc[:,'batch'] = ['rna']*N + ['atac']*N
adata.obsm['mu'] = mu 
adata.obsm['embed'] = mu 

sc.pp.neighbors(adata, use_rep='mu')
sc.tl.umap(adata, min_dist = 0.1)
sc.pl.umap(adata, color='batch')



In [ ]:
sc.pp.neighbors(adata, use_rep='embed')
sc.tl.umap(adata, min_dist = 0.1)
sc.pl.umap(adata, color='batch')
